# Module 5: Multimodal Supplier Risk Intelligence (Capstone)

Qdrant Beginners Course, follow-along notebook.

Course page: https://qdrant.tech/course/beginners/module-5/

## Recap: Modules 1-4

Embeddings + cosine similarity for meaning. Collection/point/vector/payload data model, HNSW, payload filters. Hybrid search (dense + sparse via Prefetch + fusion). Five system layers (Query, Indexing, Storage, Data, Distribution), decide the data layer before ingesting.

This module: one collection, three named vectors (text dense, text sparse, image), clustering, and multimodal queries.


In [ ]:
!pip install -q "qdrant-client[fastembed]" scikit-learn numpy

## Project

A factory fire reaches you as a news report, a satellite image, an earnings-call remark, and a filing, none labeled as an incident. Build one system: ingest daily, cluster signals into events, query every modality.

## Architecture

1. **Ingest**: collect the day's signals, chunk long text.
2. **Embed**: `text_dense`, `text_sparse`, `image` named vectors.
3. **Store**: one `PointStruct` per signal, carrying whatever vectors it has, plus payload (supplier, source type, country, date, risk score).
4. **Cluster and query**: daily batch tags a `cluster_id`, analysts run hybrid + image queries.

```yaml
collection: supplier_signals

named_vectors:
  text_dense:  { model: all-MiniLM-L6-v2, size: 384, distance: Cosine }
  text_sparse: { model: Qdrant/bm25, modifier: IDF }
  image:       { model: Qdrant/clip-ViT-B-32-vision, size: 512, distance: Cosine }

payload_fields:
  supplier_id, source_type, language, country, facility_id: keyword, indexed
  published_at: datetime, indexed
  risk_score: float, indexed
  cluster_id: integer, indexed (assigned after ingestion)
  summary: text, not indexed
```

Three named vectors, not five: a transcript is text once transcribed, a video frame is an image once sampled, both reuse existing spaces.

## Models


In [ ]:
from qdrant_client import QdrantClient, models

DENSE_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"   # 384 dims
SPARSE_MODEL = "Qdrant/bm25"

# CLIP: two encoders sharing one space. Images go through vision, queries through text.
IMAGE_MODEL      = "Qdrant/clip-ViT-B-32-vision"
IMAGE_TEXT_MODEL = "Qdrant/clip-ViT-B-32-text"

`all-MiniLM-L6-v2` caps at 256 tokens. Chunk anything longer:


In [ ]:
def chunk_text(text: str, size: int = 150, overlap: int = 30) -> list[str]:
    words = text.split()
    if len(words) <= size:
        return [text]
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, len(words), step)]

CLIP puts images and captions in one shared space, that's what lets a text query match an uncaptioned photo:


In [ ]:
# At ingestion, the picture becomes the `image` vector.
satellite_input = models.Image(
    image="captures/haiphong-2026-07-15.jpg",
    model=IMAGE_MODEL,
)

# At query time, use CLIP's text encoder (not DENSE_MODEL) so the query lands
# in the same space. CLIP truncates text at 77 tokens.
image_query = models.Document(
    text="smoke above factory roof",
    model=IMAGE_TEXT_MODEL,
)

In [ ]:
# Two excerpts from a quarterly call, already transcribed (swap in Whisper output).
EARNINGS_CALL_EXCERPTS = [
    "On the Haiphong question: the line was halted for four days after the fire "
    "and two of the three shifts are running again as of this week.",
    "We are not guiding to a shortage. The backlog at the port adds a week to "
    "inbound components and we have qualified a second supplier for the housing.",
]

## Ingestion pipeline

Risk scoring is a keyword baseline, the first thing to replace once you have real labels:


In [ ]:
import re

RISK_TERMS = {
    "fire": 0.9, "explosion": 0.9, "halted": 0.8, "shutdown": 0.8,
    "recall": 0.7, "strike": 0.7, "flood": 0.7,
    "investigation": 0.5, "shortage": 0.5, "delay": 0.5,
    "backlog": 0.4, "inspection": 0.4,
}

def score_risk(text: str) -> float:
    lowered = text.lower()
    return max(
        (weight for term, weight in RISK_TERMS.items()
         if re.search(rf"\b{term}\b", lowered)),
        default=0.0,
    )

In [ ]:
import uuid
from qdrant_client import QdrantClient, models

client = QdrantClient(
    url="https://YOUR-CLUSTER.cloud.qdrant.io",
    api_key="YOUR_API_KEY",
)

client.create_collection(
    collection_name="supplier_signals",
    vectors_config={
        "text_dense": models.VectorParams(size=384, distance=models.Distance.COSINE),
        "image":      models.VectorParams(size=512, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
        "text_sparse": models.SparseVectorParams(modifier=models.Modifier.IDF),
    },
)

for field in ["supplier_id", "source_type", "language", "country", "facility_id"]:
    client.create_payload_index(
        collection_name="supplier_signals",
        field_name=field,
        field_schema=models.PayloadSchemaType.KEYWORD,
    )

client.create_payload_index(
    collection_name="supplier_signals",
    field_name="published_at",
    field_schema=models.PayloadSchemaType.DATETIME,
)
client.create_payload_index(
    collection_name="supplier_signals",
    field_name="risk_score",
    field_schema=models.PayloadSchemaType.FLOAT,
)
client.create_payload_index(
    collection_name="supplier_signals",
    field_name="cluster_id",     # values arrive after clustering, index now anyway
    field_schema=models.PayloadSchemaType.INTEGER,
)

One signal, one point: an image and its caption are one signal seen two ways. Put every vector describing the same thing on the same point.


In [ ]:
def ingest_signal(signal: dict) -> str:
    vectors = {}

    if signal.get("text"):
        vectors["text_dense"]  = models.Document(text=signal["text"], model=DENSE_MODEL)
        vectors["text_sparse"] = models.Document(text=signal["text"], model=SPARSE_MODEL)

    if signal.get("image_path"):
        vectors["image"] = models.Image(image=signal["image_path"], model=IMAGE_MODEL)

    point_id = str(uuid.uuid4())
    client.upsert(
        collection_name="supplier_signals",
        points=[
            models.PointStruct(
                id=point_id,
                vector=vectors,
                payload={
                    "supplier_id":  signal["supplier_id"],
                    "source_type":  signal["source_type"],
                    "language":     signal.get("language", "en"),
                    "country":      signal.get("country"),
                    "facility_id":  signal.get("facility_id"),
                    "published_at": signal["published_at"],
                    "risk_score":   score_risk(signal.get("text", "")),
                    "summary":      signal.get("text", "")[:300],
                },
            )
        ],
    )
    return point_id

def ingest_news_article(article: dict):
    for chunk in chunk_text(article["text"]):
        ingest_signal({**article, "text": chunk})

def ingest_satellite_capture(capture: dict):
    # Caption at ingestion, not later: an uncaptioned image has no text_dense
    # vector, so it can never join a text cluster or match a text query.
    ingest_signal({
        "text":         capture["caption"],
        "image_path":   capture["image_path"],
        "source_type":  "satellite",
        **{k: capture[k] for k in ("supplier_id", "facility_id", "country", "published_at")},
    })

def ingest_earnings_call(call: dict, excerpts: list[str]):
    for excerpt in excerpts:
        for chunk in chunk_text(excerpt):
            ingest_signal({**call, "text": chunk, "source_type": "audio"})

In [ ]:
ingest_earnings_call(
    {
        "supplier_id":  "SUP-7291",
        "country":      "VN",
        "facility_id":  "FAC-HAIPHONG-1",
        "published_at": "2026-07-16T14:00:00Z",
    },
    EARNINGS_CALL_EXCERPTS,
)

## Clustering risk signals

Group signals describing the same event, even across sources. Retrieve the day's vectors with `scroll`, run k-means, write a `cluster_id` back to each point. A cluster centroid is itself a vector you can query with.


In [ ]:
import numpy as np
from datetime import datetime, timedelta, timezone

def get_supplier_signals_last_24h(supplier_id: str):
    since = (datetime.now(timezone.utc) - timedelta(hours=24)).isoformat()

    scroll_filter = models.Filter(
        must=[
            models.FieldCondition(key="supplier_id", match=models.MatchValue(value=supplier_id)),
            models.FieldCondition(key="published_at", range=models.DatetimeRange(gte=since)),
        ]
    )

    points, offset = [], None
    while True:
        batch, offset = client.scroll(
            collection_name="supplier_signals",
            scroll_filter=scroll_filter,
            with_vectors=True,
            limit=256,
            offset=offset,
        )
        points.extend(batch)
        if offset is None:
            break
    return points

def dense_matrix(points):
    ids, vecs = [], []
    for p in points:
        # A signal with no text (uncaptioned image) has no text_dense vector.
        if p.vector and "text_dense" in p.vector:
            ids.append(p.id)
            vecs.append(p.vector["text_dense"])
    if not vecs:
        return [], None
    return ids, np.asarray(vecs, dtype=np.float32)

In [ ]:
from sklearn.cluster import KMeans

def cluster_and_tag(supplier_id: str, n_clusters: int = 5):
    points   = get_supplier_signals_last_24h(supplier_id)
    ids, arr = dense_matrix(points)

    if arr is None or len(ids) < n_clusters:
        return None

    model = KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit(arr)

    for label in sorted(set(int(l) for l in model.labels_)):
        client.set_payload(
            collection_name="supplier_signals",
            payload={"cluster_id": label},
            points=[pid for pid, l in zip(ids, model.labels_) if int(l) == label],
        )
    return model.cluster_centers_

def signals_like_cluster(centroid, limit: int = 20):
    return client.query_points(
        collection_name="supplier_signals",
        query=centroid.tolist(),
        using="text_dense",
        limit=limit,
        with_payload=True,
    )

centroids = cluster_and_tag("SUP-7291")
if centroids is not None:
    matches = signals_like_cluster(centroids[0])

In [ ]:
def signals_in_cluster(supplier_id: str, cluster_id: int):
    points, _ = client.scroll(
        collection_name="supplier_signals",
        scroll_filter=models.Filter(
            must=[
                models.FieldCondition(key="supplier_id", match=models.MatchValue(value=supplier_id)),
                models.FieldCondition(key="cluster_id", match=models.MatchValue(value=cluster_id)),
            ]
        ),
        limit=50,
    )
    return points

## Analyst queries

Search images with text, the query goes through CLIP's text encoder:


In [ ]:
def search_facility_images(query_text: str, supplier_id: str, limit: int = 10):
    return client.query_points(
        collection_name="supplier_signals",
        query=models.Document(text=query_text, model=IMAGE_TEXT_MODEL),
        using="image",
        query_filter=models.Filter(
            must=[models.FieldCondition(key="supplier_id", match=models.MatchValue(value=supplier_id))]
        ),
        limit=limit,
        with_payload=True,
    )

smoke = search_facility_images("smoke above factory roof", supplier_id="SUP-7291")

Hybrid investigation query, filter inside each `Prefetch` as in Modules 3-4:


In [ ]:
def query_supplier_risk(supplier_id: str, query_text: str):
    cutoff = (datetime.now(timezone.utc) - timedelta(days=7)).isoformat()

    risk_filter = models.Filter(
        must=[
            models.FieldCondition(key="supplier_id", match=models.MatchValue(value=supplier_id)),
            models.FieldCondition(key="risk_score", range=models.Range(gte=0.5)),
            models.FieldCondition(key="published_at", range=models.DatetimeRange(gte=cutoff)),
        ]
    )

    return client.query_points(
        collection_name="supplier_signals",
        prefetch=[
            models.Prefetch(
                query=models.Document(text=query_text, model=DENSE_MODEL),
                using="text_dense", filter=risk_filter, limit=50,
            ),
            models.Prefetch(
                query=models.Document(text=query_text, model=SPARSE_MODEL),
                using="text_sparse", filter=risk_filter, limit=50,
            ),
        ],
        query=models.RrfQuery(rrf=models.Rrf()),
        limit=10,
    )

### Cross-language

Point `DENSE_MODEL` at a multilingual model such as `intfloat/multilingual-e5-large` (100 languages, 1024 dims, needs `query:` / `passage:` prefixes) to reach non-English sources. Filter by `language` to compare coverage.

### Try it yourself

1. Ingest a satellite capture with an empty caption, confirm it never joins a cluster.
2. Add a `source_type` filter argument to `query_supplier_risk`.
3. Run `search_facility_images` with `IMAGE_TEXT_MODEL` vs `DENSE_MODEL` and compare.

## Course summary

| Module | Theme |
|--------|-------|
| 1 | Semantic search, embeddings, cosine similarity |
| 2 | Collections, points, HNSW, filtering, chunking |
| 3 | Sparse vs dense vs hybrid search |
| 4 | Designing a system that scales |
| 5 | Multimodal capstone: ingest, cluster, query |

## Further reading

- [Named Vectors](https://qdrant.tech/documentation/manage-data/vectors/#named-vectors)
- [Hybrid Queries](https://qdrant.tech/documentation/search/hybrid-queries/)
- [Indexing and Filterable HNSW](https://qdrant.tech/documentation/manage-data/indexing/)
- [FastEmbed](https://qdrant.tech/documentation/fastembed/)
- [multilingual-e5-large](https://huggingface.co/intfloat/multilingual-e5-large)
- [CLIP ViT-B/32](https://huggingface.co/openai/clip-vit-base-patch32)

Course complete: Modules 1 through 5.
